# BERT-Tiny E4 Indian Language Identification Benchmark
The notebook audits the supplied exports, reconciles predictions by `eval_id`, calculates full-set and paired-subset metrics, saves CSVs and high-resolution charts. The MIRA export is a reference proxy because the original held-out dataset was not separately uploaded.

In [ ]:
# BERT-Tiny E4 — Indian Language Identification Benchmark
# Upload the four supplied prediction exports when prompted.
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from google.colab import files
uploaded = files.upload()
def find_file(fragment):
    matches=[k for k in uploaded if fragment.lower() in k.lower()]
    if not matches: raise FileNotFoundError(f"Upload a file with name containing: {fragment}")
    return matches[0]
bert=pd.read_csv(find_file("bert_tinypredictioncomparision"))
mira=pd.read_csv(find_file("MIRA_IndicLID"))
fast=pd.read_csv(find_file("fasttextpredictioncomparision"))
sarvam=pd.read_csv(find_file("sarvam vs bert"))
OUT="benchmark_outputs"; os.makedirs(OUT,exist_ok=True)
LABELS=["as-IN","bn-IN","en-IN","gu-IN","hi-IN","kn-IN","ml-IN","mr-IN","pa-IN","ta-IN","te-IN"]
NAMES={"as-IN":"Assamese","bn-IN":"Bengali","en-IN":"English","gu-IN":"Gujarati","hi-IN":"Hindi",
       "kn-IN":"Kannada","ml-IN":"Malayalam","mr-IN":"Marathi","pa-IN":"Punjabi","ta-IN":"Tamil","te-IN":"Telugu"}
def norm(x):
    if pd.isna(x): return "__missing__"
    x=str(x).strip()
    return x if x in LABELS else "__unsupported__"

# PHASE 1: dataset audit. MIRA export is a proxy reference because the original held-out dataset was not separately uploaded.
print("MIRA reference shape:",mira.shape)
print("Schema:\n",mira.dtypes)
print("Missing values:\n",mira.isna().sum())
mira["text_length_chars"]=mira.text.astype(str).str.len()
mira["text_length_words_whitespace"]=mira.text.astype(str).str.split().str.len()
audit_summary=pd.DataFrame([
    ["rows",len(mira)],["columns",len(mira.columns)],["unique_eval_id",mira.eval_id.nunique()],
    ["missing_eval_id",mira.eval_id.isna().sum()],["duplicate_eval_id",mira.eval_id.duplicated().sum()],
    ["duplicate_full_rows",mira.duplicated().sum()],["duplicate_text_exact",mira.text.duplicated().sum()],
    ["empty_or_whitespace_text",(mira.text.astype(str).str.strip()=="").sum()],
    ["text_length_le_5_chars",(mira.text_length_chars<=5).sum()],["text_length_gt_500_chars",(mira.text_length_chars>500).sum()],
    ["missing_values_any_column",int(mira.isna().sum().sum())],
    ["majority_class_baseline_accuracy",mira.label.value_counts(normalize=True).max()]
],columns=["audit_item","value"])
audit_summary.to_csv(f"{OUT}/dataset_audit_summary.csv",index=False)
mira.dtypes.astype(str).rename("dtype").to_csv(f"{OUT}/dataset_schema.csv")
pd.DataFrame({"column":mira.columns,"missing_count":mira.isna().sum().values,
              "missing_percent":mira.isna().mean().values*100}).to_csv(f"{OUT}/missing_values.csv",index=False)
label_dist=mira.label.value_counts().reindex(LABELS,fill_value=0).rename_axis("label").reset_index(name="count")
label_dist["language"]=label_dist.label.map(NAMES); label_dist["percent"]=label_dist["count"]/len(mira)*100
label_dist.to_csv(f"{OUT}/ground_truth_language_distribution.csv",index=False)
for col in ["source_dataset","script","source_type","variant","category","language_detail"]:
    if col in mira:
        mira.groupby([col,"label"],dropna=False).size().rename("count").reset_index().to_csv(f"{OUT}/distribution_by_{col}.csv",index=False)
print("Label distribution:\n",label_dist)

# PHASE 2: reconcile strictly by eval_id; preserve raw outputs and normalize only supported labels.
for name,df in [("BERT",bert),("MIRA",mira),("fast-langdetect",fast)]:
    assert df.eval_id.is_unique, f"{name}: eval_id is not unique"
assert set(bert.eval_id)==set(mira.eval_id)==set(fast.eval_id), "Full-set ID coverage mismatch"
check=bert[["eval_id","text","label"]].merge(mira[["eval_id","text","label"]],on="eval_id",suffixes=("_bert","_mira"),validate="one_to_one")
assert (check.text_bert==check.text_mira).all() and (check.label_bert==check.label_mira).all()
check=fast[["eval_id","text_x","label_x"]].merge(mira[["eval_id","text","label"]],on="eval_id",validate="one_to_one")
assert (check.text_x==check.text).all() and (check.label_x==check.label).all()
master=mira[["eval_id","text","label","category","script","source_type","variant","source_dataset","hard_negative","n_chars","n_words"]].copy()
master=master.merge(bert[["eval_id","bert_pred","bert_confidence","bert_latency_ms"]],on="eval_id",validate="one_to_one")
master=master.merge(fast[["eval_id","fastlangdetect_raw","fastlangdetect_pred","confidence","latency_ms"]].rename(columns={"confidence":"fastlangdetect_confidence","latency_ms":"fastlangdetect_latency_ms"}),on="eval_id",validate="one_to_one")
master=master.merge(mira[["eval_id","pred_MIRA_IndicLID","confidence_MIRA_IndicLID","latency_ms_MIRA_IndicLID"]],on="eval_id",validate="one_to_one")
master["bert_norm"]=master.bert_pred.map(norm)
master["fast_norm"]=master.fastlangdetect_pred.map(norm)
master["mira_norm"]=master.pred_MIRA_IndicLID.map(norm)
master.to_csv(f"{OUT}/master_full_dataset_comparison.csv",index=False)
pd.DataFrame([
["BERT-Tiny E4","Native target labels","Other labels -> __unsupported__"],
["fast-langdetect","fastlangdetect_raw uses language codes; fastlangdetect_pred has normalized tags","Only exact target tags retained; other labels -> __unsupported__"],
["MIRA-IndicLID","pred_MIRA_IndicLID","Only exact target tags retained; other labels -> __unsupported__"],
["Sarvam /text-lid","sarvam_prediction","od-IN and __unsupported__ treated as unsupported for this 11-label target"]
],columns=["model","observed_format","normalization_policy"]).to_csv(f"{OUT}/label_mapping_audit.csv",index=False)

def calc(y,p,name,latency=None):
    y=pd.Series(y).astype(str).reset_index(drop=True); p=pd.Series(p).astype(str).reset_index(drop=True)
    ok=p.isin(LABELS); yy=y[ok]; pp=p[ok]
    pr,re,f1,sup=precision_recall_fscore_support(yy,pp,labels=LABELS,zero_division=0)
    wp,wr,wf,_=precision_recall_fscore_support(yy,pp,labels=LABELS,average="weighted",zero_division=0)
    r={"model":name,"n_rows":len(y),"successful_supported_predictions":int(ok.sum()),
       "unsupported_predictions":int((p=="__unsupported__").sum()),"missing_predictions":int((p=="__missing__").sum()),
       "coverage_percent":ok.mean()*100,"accuracy_successful_predictions_percent":accuracy_score(yy,pp)*100,
       "operational_accuracy_all_rows_unsupported_wrong_percent":(p.values==y.values).mean()*100,
       "balanced_accuracy_successful_predictions_percent":balanced_accuracy_score(yy,pp)*100,
       "macro_precision_successful_percent":pr.mean()*100,"macro_recall_successful_percent":re.mean()*100,
       "macro_f1_successful_percent":f1.mean()*100,"weighted_precision_successful_percent":wp*100,
       "weighted_recall_successful_percent":wr*100,"weighted_f1_successful_percent":wf*100}
    if latency is not None:
        z=pd.to_numeric(pd.Series(latency),errors="coerce").dropna()
        r.update({"latency_n":len(z),"latency_mean":z.mean(),"latency_std":z.std(),"latency_min":z.min(),
                  "latency_p50":z.quantile(.5),"latency_p90":z.quantile(.9),"latency_p95":z.quantile(.95),
                  "latency_max":z.max(),"throughput_from_p50_1_per_second":1000/z.quantile(.5) if z.quantile(.5)>0 else np.nan})
    return r

full_specs=[
 ("BERT-Tiny E4","bert_norm","bert_latency_ms"),
 ("fast-langdetect (uploaded implementation)","fast_norm","fastlangdetect_latency_ms"),
 ("MIRA-IndicLID","mira_norm","latency_ms_MIRA_IndicLID")]
full_metrics=pd.DataFrame([calc(master.label,master[p],name,master[lat]) for name,p,lat in full_specs])
full_metrics.to_csv(f"{OUT}/full_dataset_model_metrics.csv",index=False)
per=[]
for name,pcol,_ in full_specs:
    p=master[pcol]; ok=p.isin(LABELS)
    rep=classification_report(master.loc[ok,"label"],p[ok],labels=LABELS,output_dict=True,zero_division=0)
    for lab in LABELS:
        per.append({"model":name,"label":lab,"language":NAMES[lab],"precision":rep[lab]["precision"],
                    "recall":rep[lab]["recall"],"f1":rep[lab]["f1-score"],"support":rep[lab]["support"],
                    "false_positive_count":int(((p[ok]==lab)&(master.loc[ok,"label"]!=lab)).sum()),
                    "false_negative_count":int(((master.loc[ok,"label"]==lab)&(p[ok]!=lab)).sum())})
per=pd.DataFrame(per); per.to_csv(f"{OUT}/per_language_metrics_full.csv",index=False)
display(full_metrics)

# PHASE 3: confusion matrices, including unsupported/missing outputs as explicit prediction columns.
cm_labels=LABELS+["__unsupported__","__missing__"]
for name,pcol,_ in full_specs:
    cm=confusion_matrix(master.label,master[pcol],labels=cm_labels)
    pd.DataFrame(cm,index=cm_labels,columns=cm_labels).to_csv(f"{OUT}/confusion_matrix_{name.split()[0].lower()}.csv")
    fig,ax=plt.subplots(figsize=(10,8)); im=ax.imshow(cm,aspect="auto")
    ax.set_title("Confusion Matrix — "+name); ax.set_xlabel("Predicted label"); ax.set_ylabel("Ground truth")
    ax.set_xticks(range(len(cm_labels))); ax.set_xticklabels(cm_labels,rotation=55,ha="right")
    ax.set_yticks(range(len(cm_labels))); ax.set_yticklabels(cm_labels); fig.colorbar(im,ax=ax,label="Examples")
    fig.tight_layout(); fig.savefig(f"{OUT}/confusion_{name.split()[0].lower()}.png",dpi=220); plt.show()

# Charts
fig,ax=plt.subplots(figsize=(10,5)); ax.bar(label_dist.language,label_dist["count"])
ax.set_title("Ground-truth Language Distribution"); ax.set_xlabel("Language"); ax.set_ylabel("Examples")
ax.tick_params(axis="x",rotation=35); fig.tight_layout(); fig.savefig(f"{OUT}/01_ground_truth_language_distribution.png",dpi=220); plt.show()
fig,ax=plt.subplots(figsize=(10,5)); x=np.arange(len(full_metrics)); w=.24
for i,(col,label) in enumerate([("accuracy_successful_predictions_percent","Accuracy supported"),("macro_f1_successful_percent","Macro-F1 supported"),("operational_accuracy_all_rows_unsupported_wrong_percent","Operational accuracy")]):
    ax.bar(x+(i-1)*w,full_metrics[col],w,label=label)
ax.set_xticks(x); ax.set_xticklabels(full_metrics.model,rotation=15,ha="right"); ax.set_ylim(0,105); ax.set_ylabel("Percent"); ax.set_title("Full-set Classification Quality"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/02_full_set_metrics_comparison.png",dpi=220); plt.show()
fig,ax=plt.subplots(figsize=(10,5))
for name,g in per.groupby("model"): ax.plot(g.language,g.f1*100,marker="o",label=name)
ax.set_ylim(0,105); ax.set_ylabel("F1 (%)"); ax.set_title("Per-language F1"); ax.tick_params(axis="x",rotation=35); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/03_per_language_f1.png",dpi=220); plt.show()
fig,ax=plt.subplots(figsize=(10,5))
for i,(col,label) in enumerate([("latency_p50","p50"),("latency_p90","p90"),("latency_p95","p95")]):
    ax.bar(x+(i-1)*w,full_metrics[col],w,label=label)
ax.set_xticks(x); ax.set_xticklabels(full_metrics.model,rotation=15,ha="right"); ax.set_ylabel("Recorded latency value; verify units"); ax.set_title("Latency Percentiles"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/04_latency_percentiles_full.png",dpi=220); plt.show()

# PHASE 4: paired BERT versus Sarvam subset, no cross-track ranking against full-set results.
assert sarvam.eval_id.is_unique
assert (sarvam.text_x==sarvam.text_y).all() and (sarvam.label_x==sarvam.label_y).all()
sarvam["bert_norm"]=sarvam.bert_pred.map(norm); sarvam["sarvam_norm"]=sarvam.sarvam_prediction.map(norm)
sarvam.to_csv(f"{OUT}/bert_sarvam_subset_reconciled.csv",index=False)
subset_metrics=pd.DataFrame([
 calc(sarvam.label_x,sarvam.bert_norm,"BERT-Tiny E4 (Sarvam common subset)",sarvam.bert_latency_ms),
 calc(sarvam.label_x,sarvam.sarvam_norm,"Sarvam /text-lid API",sarvam.sarvam_latency_ms)])
subset_metrics.to_csv(f"{OUT}/bert_vs_sarvam_subset_metrics.csv",index=False)
display(subset_metrics)
print("Sarvam prediction values:\n",sarvam.sarvam_prediction.value_counts(dropna=False))
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(subset_metrics.model,subset_metrics.operational_accuracy_all_rows_unsupported_wrong_percent)
ax.set_ylim(0,105); ax.set_ylabel("Percent"); ax.set_title("BERT vs Sarvam — Same 3,952 Rows"); ax.tick_params(axis="x",rotation=10)
fig.tight_layout(); fig.savefig(f"{OUT}/05_bert_sarvam_operational_accuracy.png",dpi=220); plt.show()
fig,ax=plt.subplots(figsize=(8,5)); xx=np.arange(len(subset_metrics)); ww=.36
ax.bar(xx-ww/2,subset_metrics.latency_p50,ww,label="p50"); ax.bar(xx+ww/2,subset_metrics.latency_p90,ww,label="p90")
ax.set_xticks(xx); ax.set_xticklabels(subset_metrics.model,rotation=10); ax.set_ylabel("Recorded latency value; verify units"); ax.set_title("BERT vs Sarvam Latency"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/06_bert_sarvam_latency.png",dpi=220); plt.show()

# Error examples and frequent confusion pairs
err=master[["eval_id","text","label","script","source_type","variant","source_dataset","bert_norm","fast_norm","mira_norm"]].copy()
err["bert_correct"]=err.bert_norm==err.label; err["fast_correct"]=err.fast_norm==err.label; err["mira_correct"]=err.mira_norm==err.label
err[err.bert_correct & ~err.fast_correct].head(500).to_csv(f"{OUT}/examples_bert_correct_fastlangdetect_wrong.csv",index=False)
err[~err.bert_correct & err.fast_correct].head(500).to_csv(f"{OUT}/examples_fastlangdetect_correct_bert_wrong.csv",index=False)
err[~err.bert_correct & ~err.fast_correct & ~err.mira_correct].head(500).to_csv(f"{OUT}/examples_all_three_not_correct.csv",index=False)
pairs=[]
for name,pcol,_ in full_specs:
    q=master.loc[master[pcol].isin(LABELS),["label",pcol]].copy()
    q=q[q.label!=q[pcol]].value_counts().head(20)
    for (true,pred),count in q.items(): pairs.append({"model":name,"true_label":true,"predicted_label":pred,"count":int(count)})
pd.DataFrame(pairs).to_csv(f"{OUT}/top_confusion_pairs.csv",index=False)
print("Outputs saved to",OUT)
print("Caveats: the fast baseline is fast-langdetect according to its columns, not verified fastText lid.176; MIRA has no en-IN predictions in this export; verify latency units/timing protocol, data provenance, licensing, artifact sizes and hardware before public claims.")
